# RAG по стенограммам ПМЭФ

In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv
import pandas as pd
from IPython.display import JSON, display

PROJECT = Path.cwd()
load_dotenv(PROJECT / ".env")
if not os.getenv("CLOUD_RU_API_KEY"):
    raise ValueError("Добавьте CLOUD_RU_API_KEY в .env в корне проекта")
pd.set_option("display.max_colwidth", None)

## 1. Данные

Документ — стенограмма. Фрагмент — строка с таймкодом. [knowledge.py](knowledge.py)

In [2]:
from knowledge import load_documents, split_documents

documents = load_documents(PROJECT / "data/SPIEF_txt")
chunks = split_documents(documents)
print(f"{len(documents)} документов, {len(chunks)} фрагментов")
print(chunks[0].text[:300])

220 документов, 9565 фрагментов
[00:00:00 - 00:01:00] | М. Монти | Добрый день, дамы и господа. Меня зовут Марио Монти. Я президент университета Боккони и председатель Европейской группы Трехсторонней комиссии. В течение десяти лет я был членом Европейской комиссии, где отвечал за политические решения, находящиеся в центре внимани


## 2. Поиск

BM25 ищет по словам; Giga Embeddings — по близости векторов.
Кэшируем эмбеддинги. [search.py](search.py), [semantic.py](semantic.py)

In [3]:
from contextlib import closing
from embeddings import EmbeddingStore, create_client
from search import LexicalSearch
from semantic import SemanticSearch

question = "Какой теме был посвящён IT-завтрак на ПМЭФ-2016?"
lexical = LexicalSearch(chunks)
with create_client(PROJECT) as client:
    with closing(EmbeddingStore(PROJECT / ".local/embeddings-giga-3b.sqlite", client)) as store:
        semantic = SemanticSearch.build(chunks, store)
        hits = {"BM25": lexical.search(question, top_k=3),
                "Giga Embeddings": semantic.search(question, top_k=3)}
pd.DataFrame(
    [(method, hit.chunk.chunk_id, hit.chunk.text[:180])
     for method, found in hits.items() for hit in found],
    columns=["Поиск", "Фрагмент", "Начало текста"],
)

/Users/kpaq/Documents/code/llm-opt/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Поиск,Фрагмент,Начало текста
0,BM25,TRANSFER_2014_05_24_OC_it.txt#0014,"[00:13:00 - 00:14:00] | С. Плуготаренко | Я думаю, что наш завтрак посвящен и этому вопросу. Он проводится, чтобы мы друг друга увидели и продемонстрировали свои успехи. Анатолий,"
1,BM25,TRANSFER_2016_06_18_OC_it.txt#0056,"[00:55:00 - 00:56:00] | С. Плуготаренко | Огромное спасибо. Я надеюсь, что мы уложились теми темами, которые планировали. Оргкомитет попросил меня проанонсировать, что все желающие"
2,BM25,TRANSFER_2016_06_18_OC_it.txt#0037,"[00:36:00 - 00:37:00] | С. Плуготаренко | Огромное спасибо. Мы Вас отпускаем, а наш IT-завтрак продолжается. У нас третий блок."
3,Giga Embeddings,TRANSFER_2016_06_18_OC_it.txt#0056,"[00:55:00 - 00:56:00] | С. Плуготаренко | Огромное спасибо. Я надеюсь, что мы уложились теми темами, которые планировали. Оргкомитет попросил меня проанонсировать, что все желающие"
4,Giga Embeddings,TRANSFER_2016_06_18_OC_it.txt#0001,"[00:00:00 - 00:01:00] | С. Плуготаренко | Уважаемые коллеги, я предлагаю всем занимать места: спикерам IT- завтрака согласно табличкам, которые вы видите на столах, все остальные у"
5,Giga Embeddings,TRANSFER_2016_06_18_OC_it.txt#0008,"[00:07:00 - 00:08:00] | И. Агамирзян | Спасибо большое, Сергей. Доброе утро, коллеги. Хочу, во-первых, от лица РВК, как организатора сегодняшнего IT-завтрака, поприветствовать всех"


## 3. Ответ

`GigaChat/GigaChat-3-Pro` сам вызывает оба поиска. Возвращает JSON с ответом и цитатами.
[assistant.py](assistant.py), [app.py](app.py)

In [4]:
from assistant import BaselineConfig
from app import run_demo

demo, demo_path = run_demo(PROJECT, BaselineConfig(), [question])
if demo["status"] != "complete":
    raise RuntimeError(f"Запрос не завершён: {demo_path}")
example = demo["cases"][0]
display(JSON(example["answer"], expanded=True))
pd.DataFrame(
    [(call["name"], call["query"]) for call in example["tool_calls"]],
    columns=["Инструмент", "Запрос модели"],
)

<IPython.core.display.JSON object>

,Инструмент,Запрос модели
0,lexical_search,IT-завтрак ПМЭФ 2016 тема
1,semantic_search,тема IT-завтрака на ПМЭФ 2016


## 4. Проверка

В [golden.csv](data/golden.csv) — 15 вопросов. Сначала проверим поиск: Hit@5 — доля попаданий в нужный фрагмент среди первых пяти результатов.

In [5]:
from evaluation import run_retrieval_comparison

retrieval, retrieval_path = run_retrieval_comparison(PROJECT, chunks, lexical)
pd.DataFrame(retrieval["summary"])[["method", "hit_at_k", "mrr_at_k", "errors"]]

,method,hit_at_k,mrr_at_k,errors
0,BM25,0.6,0.513333,0
1,Giga Embeddings,0.8,0.594444,0


Теперь заново получим 15 ответов baseline. Эталоны модель не видит. [baseline.py](baseline.py)

In [6]:
from baseline import run_baseline

baseline, baseline_path = run_baseline(PROJECT, chunks, lexical)
if baseline["status"] != "complete":
    raise RuntimeError(f"Baseline не завершён: {baseline_path}")

baseline: вопрос 1/15 — ok


baseline: вопрос 2/15 — ok


baseline: вопрос 3/15 — ok


baseline: вопрос 4/15 — ok


baseline: вопрос 5/15 — ok


...

baseline: вопрос 11/15 — ok


baseline: вопрос 12/15 — ok


baseline: вопрос 13/15 — ok


baseline: вопрос 14/15 — ok


baseline: вопрос 15/15 — ok


Судья `deepseek-ai/DeepSeek-V4.1-Flash` проходит пять контрольных примеров, затем проверяет факты и поддержку цитатами. [quality.py](quality.py)

In [7]:
from quality import run_quality_evaluation

quality, quality_path = run_quality_evaluation(PROJECT, baseline_path, chunks)
if quality["status"] != "complete":
    raise RuntimeError(f"Оценка не завершена: {quality_path}")
print("Контроли судьи:", quality["calibration"]["status"])
metrics = quality["efficiency"]
pd.DataFrame([{
    "Вопросов": metrics["questions"],
    "Верных": metrics["quality"]["content_correct"],
    "Верных с цитатами": metrics["quality"]["supported_correct"],
    "Токены": metrics["assistant_chat"]["total_tokens"],
}])

Контроли судьи: passed


,Вопросов,Верных,Верных с цитатами,Токены
0,15,12,10,159734


## 5. Оптимизация

Сравниваем baseline с девятью вариантами на одних и тех же 15 вопросах:

- **Токены:** `deduplicated` убирает повторные фрагменты, `focused` оставляет по два окна текста, `focused_evidence` дополнительно сокращает промпт, `top_k_3` берёт по три результата, `compact` убирает повторы и лишние метаданные.
- **Рассуждение:** `self_consistency` независимо строит три ответа и выбирает большинство среди эквивалентных; `tree_of_thought` строит шесть гипотез на двух уровнях дерева, оценивает их и отсекает ветви.
- **Управление агентом:** `forced_retrieval` запускает оба поиска из приложения, не делегируя выбор инструментов модели; `plan_execute` сначала составляет поисковый план, затем выполняет его.

Baseline использует function calling: модель сама вызывает оба поиска. Каждый вариант заново проходит генерацию и оценку судьёй.
[context.py](context.py), [experiments.py](experiments.py)

In [8]:
from experiments import ExecutionConfig, load_selected_strategy, run_experiments

experiments, experiments_path = run_experiments(
    PROJECT, chunks, lexical, baseline_path, quality_path,
)
comparison = experiments["comparison"]

def comparison_row(run: dict, baseline_tokens: int, winner: str | None) -> dict:
    """Показать фактический расход и известные оценки, не считая неизвестные ошибками."""
    efficiency = run.get("efficiency", {})
    quality = efficiency.get("quality", {})
    chat = efficiency.get("assistant_chat", {})
    tokens = chat.get("total_tokens")

    def count(key: str, unknown_key: str) -> str:
        """Различать известные оценки и неизвестные результаты судьи."""
        if key not in quality:
            return "нет данных"
        known, unknown = quality[key], quality.get(unknown_key, 0)
        return f"≥{known} (неизвестно: {unknown})" if unknown else str(known)

    savings = 100 * (baseline_tokens - tokens) / baseline_tokens if tokens is not None and baseline_tokens else None
    return {
        "Вариант": run["name"] + (" ★" if run["name"] == winner else ""),
        "Токены GigaChat": tokens,
        "Экономия, %": round(savings, 2) if savings is not None else None,
        "Верных": count("content_correct", "content_unknown"),
        "Верных с цитатами": count("supported_correct", "supported_unknown"),
        "Токены / полезный ответ": chat.get("tokens_per_supported_correct_answer"),
        "Причина отказа": "; ".join(run.get("rejection_reasons", [])) or "—",
    }

runs = [{"name": "baseline", **comparison["baseline"]}, *comparison["experiments"]]
baseline_tokens = comparison["baseline"]["efficiency"]["assistant_chat"]["total_tokens"]
winner = comparison["selected"]["name"] if comparison["selected"] else None
pd.DataFrame([comparison_row(run, baseline_tokens, winner) for run in runs]).round(2)

deduplicated: вопрос 1/15 — ok


deduplicated: вопрос 2/15 — ok


deduplicated: вопрос 3/15 — ok


deduplicated: вопрос 4/15 — ok


deduplicated: вопрос 5/15 — ok


...

plan_execute: вопрос 12/15 — ok


plan_execute: вопрос 13/15 — ok


plan_execute: вопрос 14/15 — ok


plan_execute: вопрос 15/15 — ok


,Вариант,Токены GigaChat,"Экономия, %",Верных,Верных с цитатами,Токены / полезный ответ,Причина отказа
0,baseline,159734,0.00,12,10,15973.40,—
1,deduplicated,146319,8.40,12,11,13301.73,—
2,focused,114413,28.37,≥9 (неизвестно: 1),≥9 (неизвестно: 1),NaN,Эксперимент завершился ошибкой: failed
3,focused_evidence,112180,29.77,9,7,16025.71,Pipeline/API error или неполная трасса
4,top_k_3,119204,25.37,11,11,10836.73,Снижение content_correct
5,compact,141597,11.35,10,10,14159.70,Снижение content_correct
6,self_consistency,543808,-240.45,12,11,49437.09,Нет снижения фактических токенов ассистента
7,tree_of_thought,819818,-413.24,10,9,91090.89,Снижение content_correct; Снижение supported_correct; Нет снижения фактических токенов ассистента
8,forced_retrieval ★,135597,15.11,14,12,11299.75,—
9,plan_execute,147462,7.68,13,12,12288.50,—


Полезный ответ — верный и подтверждённый цитатами. Делим **все фактические токены GigaChat**, включая ошибки, на число таких ответов. Неизвестные оценки судьи показаны отдельно; при них отношение не вычисляется. Эмбеддинги и вызовы судьи учитываются отдельно и в эту таблицу токенов не входят.

Quality gate требует не снижать число верных и подтверждённых ответов относительно baseline и уменьшить расход; ★ отмечает самый экономичный прошедший вариант. Это модельная оценка на 15 вопросах, а не гарантия качества на любых данных или улучшения при каждом запуске.

In [9]:
selected = comparison["selected"]
if selected is None:
    config = BaselineConfig()
    execution = ExecutionConfig()
    print("В текущем запуске ни один вариант не прошёл quality gate с экономией; оставляем baseline.")
else:
    winner_spec = load_selected_strategy(PROJECT, experiments_path)
    config, execution = winner_spec.config, winner_spec.execution
    print(f"Выбран {selected['name']}: экономия {selected['token_savings_percent']:.2f}% токенов; quality gate пройден.")
print(f"Стратегия агента: {execution.agent_strategy}; рассуждение: {execution.reasoning_strategy}")
print("Отчёт текущего запуска:", experiments_path.relative_to(PROJECT))

Выбран forced_retrieval: экономия 15.11% токенов; quality gate пройден.
Стратегия агента: forced; рассуждение: direct
Отчёт текущего запуска: results/experiments_20261029T142937724241Z.json


## 6. Свой вопрос

Заново вызываем API с выбранными настройками: сначала для известного вопроса из раздела 2, затем для вопроса, на который корпус не должен отвечать. Показываем реальные запросы к поиску и их происхождение, число вызовов чата и расход. При baseline происхождение поиска — модель.

In [10]:
from efficiency import summarize_calls

def show_demo(report: dict, path: Path) -> None:
    """Показать ответы и фактическую трассу каждого вопроса."""
    if report["status"] != "complete":
        raise RuntimeError(f"Запрос не завершён: {path}")
    for case in report["cases"]:
        print("Вопрос:", case["question"])
        display(JSON(case["answer"], expanded=True))
        display(pd.DataFrame(
            [(call["name"], call.get("origin", "model"), call["query"])
             for call in case["tool_calls"]],
            columns=["Инструмент", "Источник вызова", "Запрос"],
        ))
        usage = summarize_calls(case["chat_calls"])
        print(f"Вызовов чата: {usage['api_attempts']}; токены GigaChat: {usage['total_tokens']}")
    print("Трасса:", path.relative_to(PROJECT))

new_question = "Какой серийный номер личного ноутбука автора проекта?"
result, result_path = run_demo(PROJECT, config, [question, new_question], execution=execution)
show_demo(result, result_path)

Вопрос: Какой теме был посвящён IT-завтрак на ПМЭФ-2016?


<IPython.core.display.JSON object>

,Инструмент,Источник вызова,Запрос
0,lexical_search,application,Какой теме был посвящён IT-завтрак на ПМЭФ-2016?
1,semantic_search,application,Какой теме был посвящён IT-завтрак на ПМЭФ-2016?


Вызовов чата: 1; токены GigaChat: 7525
Вопрос: Какой серийный номер личного ноутбука автора проекта?


<IPython.core.display.JSON object>

,Инструмент,Источник вызова,Запрос
0,lexical_search,application,Какой серийный номер личного ноутбука автора проекта?
1,semantic_search,application,Какой серийный номер личного ноутбука автора проекта?


Вызовов чата: 1; токены GigaChat: 9401
Трасса: results/demo_selected_20261029T144728175197Z.json


In [11]:
new_question = "Тема доклада на самите"
result, result_path = run_demo(PROJECT, config, [new_question], execution=execution)
show_demo(result, result_path)

Вопрос: Тема доклада на самите


<IPython.core.display.JSON object>

,Инструмент,Источник вызова,Запрос
0,lexical_search,application,Тема доклада на самите
1,semantic_search,application,Тема доклада на самите


Вызовов чата: 1; токены GigaChat: 4781
Трасса: results/demo_selected_20261029T144740274115Z.json
